# 0.0 量化交易的数学底座：线性代数直觉

> **“为什么我要学这个？”**
> 
> 很多量化交易的新手会觉得指标（RSI, MACD）就是全部。但当你开始处理**一篮子股票**（组合管理）或者想理解**市场背后的力量**（因子分析）时，你会发现指标不够用了。
> 
> **线性代数不是为了让你算难题，它是为了让你“批量处理”信息。**

## 学习目标
- 理解向量（Vector）和矩阵（Matrix）在交易中的真实含义
- 掌握矩阵运算在收益率计算和权重调整中的应用
- 直观理解主成分分析（PCA）如何帮我们看清市场

## 1. 向量与矩阵：从“单挑”到“群殴”

在只有一只股票时，我们处理的是标量（单个数字）。但现实中我们面对的是整个市场。

### 向量（Vector）是什么？
想象你有一个投资组合，持有了苹果（AAPL）、微软（MSFT）和谷歌（GOOG）。
你可以把它们的权重写成一个**权重向量 $w$**：
$$w = [0.4, 0.3, 0.3]$$
这代表你 40% 的钱买苹果，剩下两家各 30%。

### 矩阵（Matrix）是什么？
矩阵就是把多个向量叠在一起。比如这三只股票过去 5 天的日收益率，就是一个 $5 \times 3$ 的矩阵：

| 日期 | AAPL | MSFT | GOOG |
| :--- | :--- | :--- | :--- |
| Day 1 | 0.01 | -0.01 | 0.02 |
| Day 2 | -0.02 | 0.00 | 0.01 |
| ... | ... | ... | ... |

**核心直觉**：矩阵就是一张 Excel 表，它是量化代码处理数据的“基本单位”。

In [ ]:
import numpy as np
import pandas as pd

# 演示：用权重向量计算组合收益率
weights = np.array([0.4, 0.3, 0.3]) # 权重向量
returns = np.array([0.01, -0.02, 0.03]) # 三只股票当天的收益率向量

# 组合收益 = 权重1*收益1 + 权重2*收益2 + ...
# 在线性代数里，这就是“点积”（Dot Product）
portfolio_return = np.dot(weights, returns)

print(f"今日组合收益率: {portfolio_return:.4%}")

## 2. 矩阵乘法的奥秘：批量计算收益

如果你有 1000 天的数据和 100 只股票，你想计算这 1000 天组合每天的收益。你不需要写 `for` 循环，只需要一次矩阵乘法。

$$R_{portfolio} = R_{matrix} \times w$$

矩阵乘法的本质是：**重新线性组合**。它把原始的股票收益，按照你的权重重新“揉”成了一个新的时间序列。

### 为什么我们要强调不写循环？
1. **速度**：Python 的 `for` 循环很慢，但 NumPy 的矩阵运算背后是 C 语言优化的，速度快百倍。
2. **清晰**：一行矩阵运算能代替几十行逻辑代码。

In [ ]:
# 模拟 5 天 3 只股票的收益率矩阵
np.random.seed(42)
return_matrix = np.random.normal(0.001, 0.02, (5, 3))
print("收益率矩阵 (5天 x 3只股票):")
print(return_matrix)

# 一次性计算 5 天的组合收益
portfolio_returns_5d = np.dot(return_matrix, weights)
print("\n5天组合收益序列:")
print(portfolio_returns_5d)

## 3. 特征值与特征向量：看透市场的“骨架”

> 这是 QuantEcon 中非常精彩的部分，但我们用大白话来讲。

想象标普 500 指数里的 500 只股票。它们每天乱跳，看起来有 500 种不同的动向。但实际上，大多数股票是跟着“大盘”走的，或者跟着“科技板块”走的。

**特征分解（Eigen-decomposition）** 就像是给市场照 X 光：
- **特征向量** 代表了市场运动的“方向”（比如：全市场上涨、价值股领先、科技股崩盘）。
- **特征值** 代表了这个方向有多重要（影响力有多大）。

在量化里，我们常说的 **PCA（主成分分析）** 就是利用这个原理，把 500 个混乱的维度压缩成少数几个“核心变量”。

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import yfinance as yf

# 获取三只高相关科技股的数据
tickers = ['AAPL', 'MSFT', 'GOOGL']
data = yf.download(tickers, start='2023-01-01', end='2024-01-01', progress=False)['Close']
rets = data.pct_change().dropna()

# 使用 PCA 提取第一个主成分（通常代表“市场共同因子”）
pca = PCA(n_components=1)
rets_pca = pca.fit_transform(rets)

plt.figure(figsize=(12, 6))
plt.plot(rets.index, rets.mean(axis=1), label='三只股票平均收益', alpha=0.5)
plt.plot(rets.index, rets_pca, label='第一主成分 (PCA1)', linewidth=2, color='red')
plt.title("PCA 提取的市场共同动力")
plt.legend()
plt.show()

print(f"第一主成分解释了这三只股票变动的 {pca.explained_variance_ratio_[0]:.2%}")
print("这说明这三只股票虽然名字不同，但 80% 以上的时间是在跳同一支舞。")

## 🎯 小结

- **矩阵和向量** 是量化的母语，掌握它们能让你从“算个股”升级到“管组合”。
- **矩阵乘法** 是为了把计算变快、变批量的艺术。
- **特征值分解** 揭示了复杂数据背后的简单规律（主成分）。

---
**下一节** → `01_price_and_return.ipynb`（我们将把这些数学应用到真实的价格计算中）